In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import json
import math
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

# =========================================================
# CONFIG
# =========================================================
STOCK_DIR = "/content/drive/MyDrive/FOA_Data/stock_prices"
NEWS_DIR = "/content/drive/MyDrive/FOA_Data/news"

OUTPUT_DIR = "/content/drive/MyDrive/FOA_Data/llm_forecast_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_JSONL = os.path.join(OUTPUT_DIR, "train.jsonl")
VAL_JSONL = os.path.join(OUTPUT_DIR, "val.jsonl")
TEST_JSONL = os.path.join(OUTPUT_DIR, "test.jsonl")

MODEL_NAME = "ProsusAI/finbert"

HISTORY_DAYS = 30
PRED_DAYS = 1
MAX_ARTICLE_CHARS = 500
FINBERT_BATCH_SIZE = 16
FINBERT_MAX_LEN = 256

USE_RELEVANCE_FILTER = True
MIN_RELEVANCE_SCORE = 3
TOP_SHORT_TERM_NEWS = 3
TOP_LONG_TERM_NEWS = 2

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

COMPANY_KEYWORDS = {
    "AAPL": ["apple", "aapl", "iphone", "ipad", "mac", "ios", "tim cook"],
    "AMZN": ["amazon", "amzn", "aws", "prime", "andy jassy"],
    "MSFT": ["microsoft", "msft", "azure", "windows", "xbox", "satya nadella"],
    "GOOGL": ["google", "googl", "alphabet", "youtube", "android", "sundar pichai"],
    "META": ["meta", "facebook", "instagram", "whatsapp", "mark zuckerberg"],
    "TSLA": ["tesla", "tsla", "elon musk", "model 3", "model y", "cybertruck"],
    "NVDA": ["nvidia", "nvda", "gpu", "cuda", "jensen huang", "ai chip"],
}

EVENT_PATTERNS = {
    "earnings": [
        r"\bearnings\b", r"\bquarterly results\b", r"\beps\b", r"\bguidance\b",
        r"\bbeat estimates\b", r"\bmissed estimates\b"
    ],
    "analyst": [
        r"\bupgrade\b", r"\bdowngrade\b", r"\bprice target\b", r"\brating\b"
    ],
    "mna": [
        r"\bacquisition\b", r"\bmerger\b", r"\bbuyout\b", r"\btakeover\b"
    ],
    "lawsuit_reg": [
        r"\blawsuit\b", r"\bsued\b", r"\bsec\b", r"\bregulator\b",
        r"\binvestigation\b", r"\bantitrust\b"
    ],
    "product": [
        r"\blaunch\b", r"\bunveils\b", r"\bintroduces\b", r"\biphone\b",
        r"\bipad\b", r"\bmac\b"
    ],
    "macro": [
        r"\brate cut\b", r"\brate hike\b", r"\bfed\b", r"\binflation\b",
        r"\bgdp\b", r"\brecession\b"
    ]
}

MARKET_CLOSE_HOUR = 16
MARKET_CLOSE_MINUTE = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# FINBERT
# =========================================================
print("Loading FinBERT...")
finbert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
finbert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
finbert_model.eval()

# =========================================================
# HELPERS
# =========================================================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / (avg_loss + 1e-12)
    rsi = 100 - (100 / (1 + rs))
    return rsi

def next_trading_day(day, trading_dates):
    future_dates = trading_dates[trading_dates >= pd.Timestamp(day)]
    if len(future_dates) == 0:
        return pd.NaT
    return future_dates[0]

def map_news_to_trading_date(ts, trading_dates):
    ts = pd.Timestamp(ts)
    d0 = ts.normalize()
    close_ts = d0 + pd.Timedelta(hours=MARKET_CLOSE_HOUR, minutes=MARKET_CLOSE_MINUTE)

    trading_set = set(pd.DatetimeIndex(trading_dates).normalize())

    if d0 not in trading_set:
        return next_trading_day(d0, trading_dates)

    if ts > close_ts:
        return next_trading_day(d0 + pd.Timedelta(days=1), trading_dates)

    return d0

def format_num(x):
    if pd.isna(x):
        return "nan"
    return f"{float(x):.3f}"

def summarize_price_history(hist_df):
    closes = hist_df["Adj_Close"].tolist()
    nums = [format_num(x) for x in closes]
    return ", ".join(nums)

def detect_price_context(hist_df):
    last_close = hist_df["Adj_Close"].iloc[-1]
    ret_5d = hist_df["Adj_Close"].iloc[-1] / hist_df["Adj_Close"].iloc[-6] - 1 if len(hist_df) >= 6 else np.nan
    ret_20d = hist_df["Adj_Close"].iloc[-1] / hist_df["Adj_Close"].iloc[-21] - 1 if len(hist_df) >= 21 else np.nan
    rsi_14 = hist_df["rsi_14"].iloc[-1] if "rsi_14" in hist_df.columns else np.nan
    vol_20d = hist_df["vol_20d"].iloc[-1] if "vol_20d" in hist_df.columns else np.nan

    return {
        "last_close": format_num(last_close),
        "ret_5d": format_num(ret_5d),
        "ret_20d": format_num(ret_20d),
        "rsi_14": format_num(rsi_14),
        "vol_20d": format_num(vol_20d),
    }

def classify_horizon(news_date, pred_date):
    diff_days = (pd.Timestamp(pred_date).normalize() - pd.Timestamp(news_date).normalize()).days
    return "short_term" if diff_days <= 7 else "long_term"

def detect_events(text):
    text = normalize_text(text).lower()
    found = []
    for event_name, patterns in EVENT_PATTERNS.items():
        for p in patterns:
            if re.search(p, text):
                found.append(event_name)
                break
    return found

def relevance_score(row, ticker):
    text = " ".join([
        normalize_text(row.get("Article_title", "")),
        normalize_text(row.get("Article", "")),
    ]).lower()

    keywords = COMPANY_KEYWORDS.get(ticker, [ticker.lower()])
    score = 0

    for kw in keywords:
        if kw.lower() in text:
            score += 2

    if ticker.lower() in text:
        score += 2

    events = detect_events(text)
    score += len(events)

    return score

def prepare_text(row):
    title = normalize_text(row.get("Article_title", ""))
    article = normalize_text(row.get("Article", ""))[:MAX_ARTICLE_CHARS]
    return f"{title}. {article}".strip()

def finbert_batch_score(texts):
    all_probs = []

    for i in range(0, len(texts), FINBERT_BATCH_SIZE):
        batch = texts[i:i+FINBERT_BATCH_SIZE]
        enc = finbert_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=FINBERT_MAX_LEN
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            logits = finbert_model(**enc).logits.detach().cpu().numpy()

        probs = softmax(logits, axis=1)
        all_probs.append(probs)

    if len(all_probs) == 0:
        return np.zeros((0, 3))

    return np.vstack(all_probs)

def rationale_template(row, horizon):
    score = float(row["sentiment_score"])
    events = row["event_list"]

    if "earnings" in events:
        base = "Earnings-related news can quickly change investor expectations, valuation, and short-term price reactions."
    elif "analyst" in events:
        base = "Analyst actions can affect sentiment, target expectations, and near-term trading behavior."
    elif "product" in events:
        base = "Product-related news can influence future demand expectations, competitive outlook, and growth sentiment."
    elif "lawsuit_reg" in events:
        base = "Regulatory or legal news can change perceived risk, costs, and future business uncertainty."
    elif "macro" in events:
        base = "Macro news can influence discount rates, risk appetite, and broad market direction."
    else:
        base = "This news may affect investor expectations, sentiment, or company outlook."

    if horizon == "short_term":
        tail = "The effect is likely short-term because the news is recent and may directly influence near-term trading."
    else:
        tail = "The effect is treated as longer-term because it may influence expectations beyond the immediate trading window."

    if score > 0.2:
        sign = "The overall tone is positive."
    elif score < -0.2:
        sign = "The overall tone is negative."
    else:
        sign = "The overall tone is mixed or neutral."

    return f"{base} {sign} {tail}"

def build_news_block(news_df, pred_date):
    if len(news_df) == 0:
        return "No relevant news was selected before the prediction date."

    tmp = news_df.copy()
    tmp["horizon_type"] = tmp["TradingDate"].apply(lambda d: classify_horizon(pd.Timestamp(d), pred_date))

    short_df = tmp[tmp["horizon_type"] == "short_term"].copy()
    long_df = tmp[tmp["horizon_type"] == "long_term"].copy()

    short_df["rank_score"] = short_df["relevance_score"] + short_df["sentiment_score"].abs()
    long_df["rank_score"] = long_df["relevance_score"] + long_df["sentiment_score"].abs()

    short_df = short_df.sort_values("rank_score", ascending=False).head(TOP_SHORT_TERM_NEWS)
    long_df = long_df.sort_values("rank_score", ascending=False).head(TOP_LONG_TERM_NEWS)

    lines = []

    if len(short_df) > 0:
        lines.append("Short-term effect news:")
        for _, row in short_df.iterrows():
            lines.append(
                f"- Publication time: {row['Date']}; "
                f"Title: {normalize_text(row.get('Article_title', ''))}; "
                f"Summary: {normalize_text(row.get('Article', ''))[:220]}; "
                f"Rationality: {rationale_template(row, 'short_term')}"
            )

    if len(long_df) > 0:
        lines.append("Long-term effect news:")
        for _, row in long_df.iterrows():
            lines.append(
                f"- Publication time: {row['Date']}; "
                f"Title: {normalize_text(row.get('Article_title', ''))}; "
                f"Summary: {normalize_text(row.get('Article', ''))[:220]}; "
                f"Rationality: {rationale_template(row, 'long_term')}"
            )

    if not lines:
        return "No relevant news was selected before the prediction date."

    return "\n".join(lines)

def build_supplementary_block(hist_df, pred_date):
    ctx = detect_price_context(hist_df)
    dow = pred_date.day_name()
    month = pred_date.month

    return (
        f"Supplementary information: "
        f"Prediction date is {pred_date.date()} ({dow}); "
        f"Month={month}; "
        f"Last adjusted close={ctx['last_close']}; "
        f"5-day return={ctx['ret_5d']}; "
        f"20-day return={ctx['ret_20d']}; "
        f"RSI14={ctx['rsi_14']}; "
        f"20-day volatility={ctx['vol_20d']}."
    )

def build_prompt(hist_df, news_df, pred_date, ticker):
    hist_text = summarize_price_history(hist_df)
    supp_text = build_supplementary_block(hist_df, pred_date)
    news_text = build_news_block(news_df, pred_date)

    prompt = (
        f"Historical time series: {hist_text}\n"
        f"Predict the next trading-day adjusted close value for {ticker}.\n"
        f"Historical data covers {HISTORY_DAYS} trading days with daily frequency.\n"
        f"{supp_text}\n"
        f"{news_text}\n"
        f"Output only one adjusted close value with three decimal places."
    )
    return prompt

def build_target(future_df):
    return format_num(future_df["Adj_Close"].iloc[0])

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# =========================================================
# BUILD EXAMPLES FOR ALL TICKERS
# =========================================================
examples = []

stock_files = sorted([f for f in os.listdir(STOCK_DIR) if f.endswith(".csv")])
print("Stock files found:", stock_files)

for stock_file in stock_files:
    ticker = os.path.splitext(stock_file)[0]   # AAPL from AAPL.csv
    stock_path = os.path.join(STOCK_DIR, stock_file)
    news_path = os.path.join(NEWS_DIR, f"{ticker}_news.csv")

    if not os.path.exists(news_path):
        print(f"Skipping {ticker}: news file not found -> {news_path}")
        continue

    print(f"\nProcessing {ticker}...")

    # -------------------------
    # LOAD STOCK
    # -------------------------
    stock = pd.read_csv(stock_path)
    stock.columns = [c.strip().lower() for c in stock.columns]
    stock = stock.rename(columns={
        "date": "Date",
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "adj close": "Adj_Close",
        "volume": "Volume"
    })

    try:
        stock["Date"] = pd.to_datetime(stock["Date"], errors="raise")
    except Exception:
        stock["Date"] = pd.to_datetime(stock["Date"], format="%m/%d/%Y", errors="raise")

    stock = stock.sort_values("Date").reset_index(drop=True)
    stock["ret_1d"] = stock["Adj_Close"].pct_change(1)
    stock["ret_5d"] = stock["Adj_Close"].pct_change(5)
    stock["ret_20d"] = stock["Adj_Close"].pct_change(20)
    stock["vol_20d"] = stock["ret_1d"].rolling(20).std()
    stock["rsi_14"] = compute_rsi(stock["Adj_Close"], 14)

    trading_dates = pd.DatetimeIndex(stock["Date"].drop_duplicates().sort_values())

    # -------------------------
    # LOAD NEWS
    # -------------------------
    news = pd.read_csv(news_path, engine="python")
    news.columns = [c.strip() for c in news.columns]

    if "Unnamed: 0" in news.columns:
        news = news.drop(columns=["Unnamed: 0"])

    if "Date" not in news.columns:
        print(f"Skipping {ticker}: no Date column in news file")
        continue

    news["Date"] = pd.to_datetime(news["Date"], utc=True, errors="coerce").dt.tz_convert(None)
    news = news.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    if "Stock_symbol" in news.columns:
        news = news.dropna(subset=["Stock_symbol"])
        news = news[news["Stock_symbol"].astype(str).str.upper() == ticker.upper()].copy()
    else:
        news["Stock_symbol"] = ticker

    keep_cols = [c for c in ["Date", "Article_title", "Stock_symbol", "Publisher", "Author", "Article", "Url"] if c in news.columns]
    news = news[keep_cols].copy()

    if len(news) > 0:
        news["title_clean"] = news["Article_title"].fillna("").astype(str).str.strip().str.lower() if "Article_title" in news.columns else ""
        news["head_clean"] = news["Article"].fillna("").astype(str).str[:250].str.strip().str.lower() if "Article" in news.columns else ""

        if "Article_title" in news.columns:
            news = news.drop_duplicates(subset=["Date", "title_clean"])
        if "Article" in news.columns:
            news = news.drop_duplicates(subset=["Date", "head_clean"])

        drop_cols = [c for c in ["title_clean", "head_clean"] if c in news.columns]
        news = news.drop(columns=drop_cols)

        news["relevance_score"] = news.apply(lambda r: relevance_score(r, ticker), axis=1)

        if USE_RELEVANCE_FILTER:
            news = news[news["relevance_score"] >= MIN_RELEVANCE_SCORE].copy()

        news["TradingDate"] = news["Date"].apply(lambda ts: map_news_to_trading_date(ts, trading_dates))
        news = news.dropna(subset=["TradingDate"]).copy()

    # -------------------------
    # FINBERT SCORE
    # -------------------------
    if len(news) > 0:
        print(f"Scoring {len(news)} news rows with FinBERT...")
        news["finbert_text"] = news.apply(prepare_text, axis=1)
        probs = finbert_batch_score(news["finbert_text"].fillna("").astype(str).tolist())
        news["prob_positive"] = probs[:, 0]
        news["prob_negative"] = probs[:, 1]
        news["prob_neutral"] = probs[:, 2]
        news["sentiment_score"] = news["prob_positive"] - news["prob_negative"]
        news["event_list"] = news["finbert_text"].apply(detect_events)
    else:
        news = pd.DataFrame(columns=[
            "Date", "Article_title", "Stock_symbol", "Publisher", "Author",
            "Article", "Url", "relevance_score", "TradingDate", "finbert_text",
            "prob_positive", "prob_negative", "prob_neutral",
            "sentiment_score", "event_list"
        ])

    # -------------------------
    # BUILD EXAMPLES
    # -------------------------
    count_before = len(examples)

    for end_idx in range(HISTORY_DAYS - 1, len(stock) - PRED_DAYS):
        hist_df = stock.iloc[end_idx - HISTORY_DAYS + 1 : end_idx + 1].copy()
        future_df = stock.iloc[end_idx + 1 : end_idx + 1 + PRED_DAYS].copy()
        pred_date = stock.iloc[end_idx]["Date"]

        rel_news = news[news["TradingDate"] <= pred_date].copy() if len(news) > 0 else news

        prompt = build_prompt(hist_df, rel_news, pred_date, ticker)
        target = build_target(future_df)

        examples.append({
            "ticker": ticker,
            "date": str(pred_date.date()),
            "prompt": prompt,
            "target": target
        })

    print(f"Built {len(examples) - count_before} examples for {ticker}")

print("\nTotal examples:", len(examples))

# =========================================================
# TIME SPLIT
# =========================================================
examples = sorted(examples, key=lambda x: (x["date"], x["ticker"]))

n = len(examples)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_data = examples[:n_train]
val_data = examples[n_train:n_train + n_val]
test_data = examples[n_train + n_val:]

print("Train:", len(train_data), "Val:", len(val_data), "Test:", len(test_data))

save_jsonl(TRAIN_JSONL, train_data)
save_jsonl(VAL_JSONL, val_data)
save_jsonl(TEST_JSONL, test_data)

print("Saved:", TRAIN_JSONL)
print("Saved:", VAL_JSONL)
print("Saved:", TEST_JSONL)

Loading FinBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stock files found: ['AAPL.csv', 'AMZN.csv']

Processing AAPL...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Scoring 8294 news rows with FinBERT...
Built 10822 examples for AAPL

Processing AMZN...
Scoring 4412 news rows with FinBERT...
Built 6670 examples for AMZN

Total examples: 17492
Train: 12244 Val: 2623 Test: 2625
Saved: /content/drive/MyDrive/FOA_Data/llm_forecast_data/train.jsonl
Saved: /content/drive/MyDrive/FOA_Data/llm_forecast_data/val.jsonl
Saved: /content/drive/MyDrive/FOA_Data/llm_forecast_data/test.jsonl


In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model, TaskType

# =========================================================
# CONFIG
# =========================================================
DATA_DIR = "/content/drive/MyDrive/FOA_Data/llm_forecast_data"
TRAIN_JSONL = os.path.join(DATA_DIR, "train.jsonl")
VAL_JSONL = os.path.join(DATA_DIR, "val.jsonl")
TEST_JSONL = os.path.join(DATA_DIR, "test.jsonl")

OUT_DIR = "/content/drive/MyDrive/FOA_Data/llm_forecaster_model_multi"

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MAX_LENGTH = 1024
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS = 10
LR = 1e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# LOAD TOKENIZER / MODEL
# =========================================================
print("Loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

# =========================================================
# LORA
# =========================================================
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =========================================================
# LOAD DATASET
# =========================================================
dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_JSONL,
        "validation": VAL_JSONL,
        "test": TEST_JSONL
    }
)

def tokenize_function(examples):
    texts = []
    for p, t in zip(examples["prompt"], examples["target"]):
        texts.append(f"### Instruction:\n{p}\n\n### Response:\n{t}")

    tok = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    labels = []
    for i in range(len(texts)):
        prefix = f"### Instruction:\n{examples['prompt'][i]}\n\n### Response:\n"
        prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
        lab = tok["input_ids"][i].copy()

        for j in range(min(len(prefix_ids), len(lab))):
            lab[j] = -100

        lab = [-100 if tok["attention_mask"][i][k] == 0 else lab[k] for k in range(len(lab))]
        labels.append(lab)

    tok["labels"] = labels
    return tok

tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# =========================================================
# TRAINING ARGS
# =========================================================
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    load_best_model_at_end=True,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL
# =========================================================
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

print("Saved model to:", OUT_DIR)

Loading tokenizer/model...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Map:   0%|          | 0/12244 [00:00<?, ? examples/s]

Map:   0%|          | 0/2623 [00:00<?, ? examples/s]

Map:   0%|          | 0/2625 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss


In [ ]:
import os
import re
import json
import math
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================================================
# CONFIG
# =========================================================
DATA_DIR = "/content/drive/MyDrive/FOA_Data/llm_forecast_data"
TEST_JSONL = os.path.join(DATA_DIR, "test.jsonl")

MODEL_DIR = "/content/drive/MyDrive/FOA_Data/llm_forecaster_model_multi"
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MAX_INPUT_LEN = 1024
MAX_NEW_TOKENS = 64

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# LOAD MODEL
# =========================================================
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.to(DEVICE)
model.eval()

# =========================================================
# HELPERS
# =========================================================
def parse_numbers(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return [float(x) for x in nums]

def mae(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))

def rmse(y_true, y_pred):
    return math.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    eps = 1e-8
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100

# =========================================================
# LOAD TEST DATA
# =========================================================
rows = []
with open(TEST_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

print("Test examples:", len(rows))

all_true = []
all_pred = []
results = []
fail_count = 0

# =========================================================
# RUN INFERENCE
# =========================================================
for i, row in enumerate(rows):
    prompt = (
        "### Instruction:\n"
        f"{row['prompt']}\n\n"
        "### Response:\n"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    generated = decoded.split("### Response:\n")[-1].strip()

    pred_nums = parse_numbers(generated)
    true_nums = parse_numbers(row["target"])

    if len(true_nums) != 1:
        continue

    true_val = true_nums[0]

    if len(pred_nums) == 0:
        fail_count += 1
        pred_val = 0.0
    else:
        pred_val = pred_nums[0]

    all_true.append(true_val)
    all_pred.append(pred_val)

    results.append({
        "ticker": row.get("ticker", ""),
        "date": row.get("date", ""),
        "true": true_val,
        "pred": pred_val,
        "generated_text": generated
    })

    if i % 10 == 0:
        print("\n--- Example", i, "---")
        print("Ticker:", row.get("ticker", ""))
        print("Date  :", row.get("date", ""))
        print("Target:", row["target"])
        print("Pred  :", f"{pred_val:.3f}")
        print("Raw   :", generated)

# =========================================================
# METRICS
# =========================================================
print("\nFailures needing fallback:", fail_count)
print("MAE :", mae(all_true, all_pred))
print("RMSE:", rmse(all_true, all_pred))
print("MAPE:", mape(all_true, all_pred))

results_df = pd.DataFrame(results)
results_path = os.path.join(DATA_DIR, "test_predictions.csv")
results_df.to_csv(results_path, index=False)

print("Saved predictions to:", results_path)
results_df.head()